# scikit-learn Lipophilicity Regression Baseline

This notebook extends the regression track to the Lipophilicity dataset using the same train-valid-test split contract as the Delaney notebooks.

Task: predict experimental lipophilicity (`exp`) from SMILES strings
Models: character n-gram TF-IDF with Ridge regression and ElasticNet

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
lipophilicity = pd.read_csv(DATA_DIR / 'Lipophilicity.csv')

text_column = 'smiles'
target_column = 'exp'

print('Shape:', lipophilicity.shape)
display(lipophilicity.head())

## Data Checks

In [ ]:
display(lipophilicity[target_column].describe())

plt.figure(figsize=(7, 4))
sns.histplot(lipophilicity[target_column], bins=30, kde=True)
plt.title('Experimental Lipophilicity Distribution')
plt.show()

In [ ]:
X = lipophilicity[text_column]
y = lipophilicity[target_column]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED
)

print('Train:', len(X_train), 'Valid:', len(X_valid), 'Test:', len(X_test))

## Models

In [ ]:
ridge_model = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=2)),
    ('model', Ridge(alpha=1.0)),
])

elasticnet_model = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=2)),
    ('model', ElasticNet(alpha=0.0005, l1_ratio=0.1, max_iter=5000, random_state=RANDOM_SEED)),
])

models = {
    'ridge': ridge_model,
    'elasticnet': elasticnet_model,
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    valid_pred = model.predict(X_valid)
    results.append({
        'model': name,
        'rmse': float(np.sqrt(mean_squared_error(y_valid, valid_pred))),
        'mae': float(mean_absolute_error(y_valid, valid_pred)),
        'r2': float(r2_score(y_valid, valid_pred)),
    })

results_df = pd.DataFrame(results).sort_values('rmse').reset_index(drop=True)
display(results_df.round(4))

In [ ]:
best_model_name = results_df.loc[0, 'model']
best_model = models[best_model_name]
test_pred = best_model.predict(X_test)

test_metrics = pd.DataFrame([
    {
        'model': best_model_name,
        'rmse': float(np.sqrt(mean_squared_error(y_test, test_pred))),
        'mae': float(mean_absolute_error(y_test, test_pred)),
        'r2': float(r2_score(y_test, test_pred)),
    }
])
display(test_metrics.round(4))

In [ ]:
plot_df = pd.DataFrame({
    'true': y_test,
    'predicted': test_pred,
})

plt.figure(figsize=(6, 6))
sns.scatterplot(data=plot_df, x='true', y='predicted', s=40)
line_min = min(plot_df['true'].min(), plot_df['predicted'].min())
line_max = max(plot_df['true'].max(), plot_df['predicted'].max())
plt.plot([line_min, line_max], [line_min, line_max], color='black', linestyle='--')
plt.title(f'Lipophilicity Test Predictions: {best_model_name}')
plt.xlabel('True')
plt.ylabel('Predicted')
plt.show()

## Comparison Note

This dataset uses SMILES-only text features, so it is a useful complement to the descriptor-driven Delaney notebook. Together they show two different regression input styles under the same split contract.